# External Dataset Evaluation for exp006_merged_all_concat**Objective:** Evaluate the trained exp006_merged_all_concat model (dual-encoder ConvNeXt-Base + CAFormer-S18 with all-concat fusion, trained on merged Kvasir-SEG + CVC-ClinicDB) on three unseen external datasets:1. **CVC-ColonDB** (ColonDB)2. **ETIS-LaribPolypDB** (ETIS)3. **CVC-300****This is an evaluation-only task.** No training, fine-tuning, or model modification is performed.## Experiment Details- **Experiment:**  (merged training on Kvasir-SEG + CVC-ClinicDB)- **Architecture:** Dual-encoder (ConvNeXt-Base  + CAFormer-S18 ) with all-concatenation fusion- **Training duration:** 30 epochs- **Existing Kvasir-SEG test Dice:** 0.9150- **Existing Kvasir-SEG test IoU:** 0.8622- **Model parameters:** ~127.78M- **Input resolution:** 256x256- **Fusion:**  (concat at all 4 stages)- **MSU:** Disabled- **Normalization:** ImageNet mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]- **Mask threshold:** >127 (applied before resizing)- **Prediction threshold:** 0.5 (after sigmoid)- **TTA:** OFF

## 1. Mount Google Drive and Initialize Paths

In [ ]:
from google.colab import driveimport osimport sysfrom pathlib import Path# Mount Google Drivedrive.mount('/content/drive')# Verify drive mountassert Path('/content/drive/MyDrive').exists(), 'Google Drive not mounted'print('Google Drive mounted')

In [ ]:
# Define all pathsRUNS_DIR = Path('/content/drive/MyDrive/polyp-seg-runs')EXPERIMENT_NAME = 'exp006_merged_all_concat'EXPERIMENT_DIR = RUNS_DIR / EXPERIMENT_NAME# External datasets on Drive (user must place them here)DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/polyp-data/raw')# Local working directories (on Colab VM)LOCAL_DATA_ROOT = Path('/content/external_datasets')LOCAL_EVAL_ROOT = Path('/content/external_eval')LOCAL_PREDICTIONS = LOCAL_EVAL_ROOT / 'predictions'LOCAL_VISUALIZATIONS = LOCAL_EVAL_ROOT / 'visualizations'LOCAL_METRICS = LOCAL_EVAL_ROOT / 'metrics'LOCAL_LOGS = LOCAL_EVAL_ROOT / 'logs'# Output directory on DriveDRIVE_OUTPUT_DIR = RUNS_DIR / 'external_evaluation' / EXPERIMENT_NAME# Create local directoriesfor d in [LOCAL_DATA_ROOT, LOCAL_EVAL_ROOT, LOCAL_PREDICTIONS, LOCAL_VISUALIZATIONS, LOCAL_METRICS, LOCAL_LOGS]:    d.mkdir(parents=True, exist_ok=True)DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)print('Paths initialized:')print('  RUNS_DIR: ' + str(RUNS_DIR))print('  EXPERIMENT_DIR: ' + str(EXPERIMENT_DIR))print('  DRIVE_DATA_ROOT: ' + str(DRIVE_DATA_ROOT))print('  LOCAL_DATA_ROOT: ' + str(LOCAL_DATA_ROOT))print('  LOCAL_EVAL_ROOT: ' + str(LOCAL_EVAL_ROOT))print('  DRIVE_OUTPUT_DIR: ' + str(DRIVE_OUTPUT_DIR))

## 2. Inspect Experiment Artifacts

In [ ]:
# Check experiment directory existsif not EXPERIMENT_DIR.exists():    raise FileNotFoundError('Experiment directory not found: ' + str(EXPERIMENT_DIR))# List contentsprint('Contents of ' + str(EXPERIMENT_DIR) + ':')for item in sorted(EXPERIMENT_DIR.iterdir()):    size = item.stat().st_size / (1024*1024) if item.is_file() else 0    if item.is_file():        print('  ' + item.name + ' (' + '{:.1f}'.format(size) + ' MB)')    else:        print('  ' + item.name + '/')# Verify required filesrequired_files = ['config_resolved.yaml', 'run_info.json', 'best.pt']for f in required_files:    path = EXPERIMENT_DIR / f    if not path.exists():        raise FileNotFoundError('Required file missing: ' + str(path))    print('OK ' + f + ' found')

## 3. Load Experiment Configuration and Checkpoint Info

In [ ]:
import yamlimport jsonimport torch# Load resolved configwith open(EXPERIMENT_DIR / 'config_resolved.yaml', 'r') as f:    cfg = yaml.safe_load(f)# Load run infowith open(EXPERIMENT_DIR / 'run_info.json', 'r') as f:    run_info = json.load(f)print('=== Experiment Configuration ===')print('Run name: ' + str(cfg['run']['name']))print('Seed: ' + str(cfg['run']['seed']))print('Image size: ' + str(cfg['data']['image_size']))print('Normalization mean: ' + str(cfg['data']['mean']))print('Normalization std: ' + str(cfg['data']['std']))print('Mask threshold: ' + str(cfg['data']['mask_threshold']))print('Encoder A: ' + str(cfg['model']['encoder_a']) + ' (pretrained: ' + str(cfg['model']['pretrained_a']) + ')')print('Encoder B: ' + str(cfg['model']['encoder_b']) + ' (pretrained: ' + str(cfg['model']['pretrained_b']) + ')')print('Fusion channels: ' + str(cfg['model']['channels']))print('Fusion mode: ' + str(cfg['model']['fusion']))print('Fusion heads: ' + str(cfg['model']['fusion_heads']))print('Pos enc 2D: ' + str(cfg['model']['pos_enc_2d']))print('MSU: ' + str(cfg['decoder']['msu']))print('MSU mode: ' + str(cfg['decoder']['msu_mode']))print('Loss: ' + str(cfg['train']['loss']))print('Batch size: ' + str(cfg['train']['batch_size']))print('
=== Run Info ===')print('Git commit: ' + str(run_info.get('git_commit')))print('Git dirty: ' + str(run_info.get('git_dirty')))print('Torch version: ' + str(run_info.get('torch')))print('Timm version: ' + str(run_info.get('timm')))print('GPU name: ' + str(run_info.get('gpu_name')))print('Params total: ' + '{:,}'.format(run_info.get('params_total', 0)))print('Best val Dice: ' + str(run_info.get('best_val_dice')))print('Cache SHA256: ' + str(run_info.get('data_format', {}).get('cache_sha256')))

## 4. Verify Model Architecture and Checkpoint

In [ ]:
# Install dependencies if neededimport importlib.utilimport subprocessrequired_packages = {    'timm': 'timm',    'albumentations': 'albumentations',    'yaml': 'pyyaml',    'cv2': 'opencv-python-headless',    'sklearn': 'scikit-learn',}missing = []for module, pkg in required_packages.items():    if importlib.util.find_spec(module) is None:        missing.append(pkg)if missing:    print('Installing: ' + str(missing))    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)else:    print('All required packages already present')# Verify torch and CUDAprint('Torch: ' + torch.__version__)print('CUDA available: ' + str(torch.cuda.is_available()))if torch.cuda.is_available():    print('GPU: ' + torch.cuda.get_device_name(0))    print('VRAM: ' + '{:.1f} GB'.format(torch.cuda.get_device_properties(0).total_memory / 1e9))

## 5. Download and Prepare External Datasets

In [ ]:
# Dataset download URLs and info# NOTE: These URLs may require manual registration. Update if needed.DATASET_SOURCES = {    'colondb': {        'name': 'CVC-ColonDB',        'url': 'https://www.kaggle.com/datasets/tawsifurrahman/cvc-colondb',        'kaggle': 'tawsifurrahman/cvc-colondb',        'expected_images': 380,        'image_ext': '.png',        'mask_ext': '.png',        'mask_suffix': '_mask',    },    'etis': {        'name': 'ETIS-LaribPolypDB',        'url': 'https://www.kaggle.com/datasets/tawsifurrahman/etis-larib-polyp-db',        'kaggle': 'tawsifurrahman/etis-larib-polyp-db',        'expected_images': 196,        'image_ext': '.png',        'mask_ext': '.png',        'mask_suffix': '_mask',    },    'cvc300': {        'name': 'CVC-300',        'url': 'https://www.kaggle.com/datasets/tawsifurrahman/cvc-300',        'kaggle': 'tawsifurrahman/cvc-300',        'expected_images': 60,        'image_ext': '.png',        'mask_ext': '.png',        'mask_suffix': '_mask',    },}print('Dataset sources configured. Note: These datasets may require manual download from Kaggle.')print('If kaggle CLI is configured, you can use: kaggle datasets download <kaggle_id>')for key, info in DATASET_SOURCES.items():    print('  ' + key + ': ' + info['name'] + ' - ' + info['url'] + ' (expected: ' + str(info['expected_images']) + ' images)')

In [ ]:
# Check if datasets exist on Drivefor key, info in DATASET_SOURCES.items():    dataset_dir = DRIVE_DATA_ROOT / key    if dataset_dir.exists():        images_dir = dataset_dir / 'images'        masks_dir = dataset_dir / 'masks'        n_images = len(list(images_dir.glob('*' + info['image_ext']))) if images_dir.exists() else 0        n_masks = len(list(masks_dir.glob('*' + info['mask_ext']))) if masks_dir.exists() else 0        print(key + ': FOUND - images: ' + str(n_images) + ', masks: ' + str(n_masks))    else:        print(key + ': NOT FOUND at ' + str(dataset_dir))

In [ ]:
# If datasets not on Drive, try downloading via Kaggle CLI (requires kaggle.json in Colab secrets)import jsonfrom google.colab import userdatadef setup_kaggle():    """Setup Kaggle credentials from Colab secrets."""    try:        kaggle_json = userdata.get('KAGGLE_JSON')        if kaggle_json:            kaggle_dir = Path('/root/.kaggle')            kaggle_dir.mkdir(exist_ok=True)            (kaggle_dir / 'kaggle.json').write_text(kaggle_json)            os.chmod(kaggle_dir / 'kaggle.json', 0o600)            print('Kaggle credentials configured')            return True    except Exception as e:        print('Kaggle credentials not found in Colab secrets: ' + str(e))    return Falsekaggle_ready = setup_kaggle()

In [ ]:
# Download datasets if not present and kaggle is configuredimport subprocessimport zipfiledef download_dataset(dataset_key, info, target_dir):    """Download and extract a dataset from Kaggle."""    if not kaggle_ready:        print('Skipping ' + dataset_key + ': Kaggle not configured')        return False        target_dir = Path(target_dir)    if target_dir.exists() and any(target_dir.iterdir()):        print(dataset_key + ': Already exists at ' + str(target_dir))        return True        print('Downloading ' + info['name'] + ' (' + dataset_key + ')...')    try:        # Download        result = subprocess.run(            ['kaggle', 'datasets', 'download', '-d', info['kaggle'], '-p', '/tmp'],            capture_output=True, text=True, timeout=300        )        if result.returncode != 0:            print('  Download failed: ' + result.stderr)            return False                # Find downloaded zip        zip_files = list(Path('/tmp').glob('*' + dataset_key + '*.zip'))        if not zip_files:            zip_files = list(Path('/tmp').glob('*.zip'))        if not zip_files:            print('  No zip file found after download')            return False            except Exception as e:        print('  Download error: ' + str(e))        return False        # Extract    zip_path = zip_files[0]    print('  Extracting ' + zip_path.name + '...')    try:        with zipfile.ZipFile(zip_path, 'r') as zf:            zf.extractall(target_dir)        print('  Extracted to ' + str(target_dir))        return True    except Exception as e:        print('  Extraction error: ' + str(e))        return False# Download each datasetfor key, info in DATASET_SOURCES.items():    dataset_dir = DRIVE_DATA_ROOT / key    download_dataset(key, info, dataset_dir)

In [ ]:
# Verify dataset structure and create preparation reportimport numpy as npfrom PIL import Imagedef analyze_dataset(dataset_key, info, dataset_dir):    """Analyze dataset structure and create report."""    dataset_dir = Path(dataset_dir)    images_dir = dataset_dir / 'images'    masks_dir = dataset_dir / 'masks'        # Handle nested directories (Kaggle often extracts with nested folders)    if not images_dir.exists():        # Search for images folder        candidates = list(dataset_dir.rglob('images'))        if candidates:            images_dir = candidates[0]            masks_dir = images_dir.parent / 'masks'        if not images_dir.exists() or not masks_dir.exists():        return {            'dataset': info['name'],            'key': dataset_key,            'status': 'missing_folders',            'images_dir': str(images_dir),            'masks_dir': str(masks_dir),        }        # Find image files    image_files = sorted(images_dir.glob('*' + info['image_ext']))    mask_files = sorted(masks_dir.glob('*' + info['mask_ext']))        # Match by stem (remove mask suffix)    image_stems = {f.stem: f for f in image_files}    mask_stems = {}    for f in mask_files:        stem = f.stem        if info['mask_suffix'] and stem.endswith(info['mask_suffix']):            stem = stem[:-len(info['mask_suffix'])]        mask_stems[stem] = f        # Find pairs    common_stems = sorted(set(image_stems.keys()) & set(mask_stems.keys()))    missing_masks = sorted(set(image_stems.keys()) - set(mask_stems.keys()))    missing_images = sorted(set(mask_stems.keys()) - set(image_stems.keys()))        # Analyze dimensions and mask values for first few samples    img_dims = []    mask_dims = []    mask_values = set()        for stem in common_stems[:10]:  # Sample first 10        try:            with Image.open(image_stems[stem]) as img:                img_dims.append(img.size)  # (W, H)            with Image.open(mask_stems[stem]) as msk:                mask_array = np.array(msk)                mask_dims.append((mask_array.shape[1], mask_array.shape[0]))  # (W, H)                mask_values.update(np.unique(mask_array).tolist())        except Exception as e:            print('  Error reading ' + stem + ': ' + str(e))        report = {        'dataset': info['name'],        'key': dataset_key,        'status': 'ok' if common_stems else 'no_pairs',        'images_dir': str(images_dir),        'masks_dir': str(masks_dir),        'n_images_found': len(image_files),        'n_masks_found': len(mask_files),        'n_valid_pairs': len(common_stems),        'missing_masks': missing_masks[:10],        'missing_images': missing_images[:10],        'sample_image_dims': img_dims,        'sample_mask_dims': mask_dims,        'mask_pixel_values': sorted(list(mask_values)),        'expected_images': info['expected_images'],    }    return report# Analyze all datasetspreparation_reports = []for key, info in DATASET_SOURCES.items():    dataset_dir = DRIVE_DATA_ROOT / key    if dataset_dir.exists():        report = analyze_dataset(key, info, dataset_dir)    else:        report = {            'dataset': info['name'],            'key': key,            'status': 'not_found',            'expected_images': info['expected_images'],        }    preparation_reports.append(report)        # Print summary    print('\n=== ' + report['dataset'] + ' (' + key + ') ===')    print('  Status: ' + report['status'])    print('  Images found: ' + str(report.get('n_images_found', 0)))    print('  Masks found: ' + str(report.get('n_masks_found', 0)))    print('  Valid pairs: ' + str(report.get('n_valid_pairs', 0)))    if report.get('missing_masks'):        print('  Missing masks: ' + str(len(report['missing_masks'])) + ' (e.g., ' + str(report['missing_masks'][:3]) + ')')    if report.get('missing_images'):        print('  Missing images: ' + str(len(report['missing_images'])) + ' (e.g., ' + str(report['missing_images'][:3]) + ')')    if report.get('sample_image_dims'):        print('  Sample image dims: ' + str(report['sample_image_dims'][:5]))    if report.get('mask_pixel_values'):        print('  Mask pixel values: ' + str(report['mask_pixel_values']))

In [ ]:
# Save preparation reportimport jsonfrom datetime import datetimereport_path = LOCAL_METRICS / 'dataset_preparation_report.json'report_data = {    'timestamp': datetime.now().isoformat(),    'experiment': EXPERIMENT_NAME,    'datasets': preparation_reports,}with open(report_path, 'w') as f:    json.dump(report_data, f, indent=2)print('Preparation report saved to: ' + str(report_path))# Also save to Drivedrive_report_path = DRIVE_OUTPUT_DIR / 'dataset_preparation_report.json'with open(drive_report_path, 'w') as f:    json.dump(report_data, f, indent=2)print('Preparation report saved to Drive: ' + str(drive_report_path))

## 6. Preprocess External Datasets (Build Cache)

In [ ]:
# Build npz cache for each external dataset using the same preprocessing as trainingimport numpy as npfrom PIL import Imagefrom tqdm.auto import tqdmIMAGE_SIZE = cfg['data']['image_size']MASK_THRESHOLD = cfg['data']['mask_threshold']NORMALIZE = cfg['data']['normalize']MEAN = np.array(cfg['data']['mean'], dtype=np.float32).reshape(3, 1, 1)STD = np.array(cfg['data']['std'], dtype=np.float32).reshape(3, 1, 1)def build_external_cache(dataset_key, info, dataset_dir, cache_dir):    """Build npz cache for external dataset."""    dataset_dir = Path(dataset_dir)    cache_dir = Path(cache_dir)    cache_dir.mkdir(parents=True, exist_ok=True)    cache_path = cache_dir / (dataset_key + '_256.npz')        # Find images and masks directories    images_dir = dataset_dir / 'images'    masks_dir = dataset_dir / 'masks'        if not images_dir.exists():        candidates = list(dataset_dir.rglob('images'))        if candidates:            images_dir = candidates[0]            masks_dir = images_dir.parent / 'masks'        if not images_dir.exists() or not masks_dir.exists():        raise ValueError('Images/masks folders not found in ' + str(dataset_dir))        # Find matching pairs    image_files = sorted(images_dir.glob('*' + info['image_ext']))    image_stems = {f.stem: f for f in image_files}        mask_files = sorted(masks_dir.glob('*' + info['mask_ext']))    mask_stems = {}    for f in mask_files:        stem = f.stem        if info['mask_suffix'] and stem.endswith(info['mask_suffix']):            stem = stem[:-len(info['mask_suffix'])]        mask_stems[stem] = f        common_stems = sorted(set(image_stems.keys()) & set(mask_stems.keys()))        if not common_stems:        raise ValueError('No matching image/mask pairs found in ' + str(dataset_dir))        print('Building cache for ' + info['name'] + ': ' + str(len(common_stems)) + ' pairs')        # Preallocate arrays    images_arr = np.empty((len(common_stems), IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)    masks_arr = np.empty((len(common_stems), IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)        for i, stem in enumerate(tqdm(common_stems, desc='Processing ' + dataset_key)):        # Load and process image        with Image.open(image_stems[stem]) as img:            img = img.convert('RGB')            img = img.resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.BILINEAR)            images_arr[i] = np.array(img, dtype=np.uint8)                # Load and process mask (threshold BEFORE resize)        with Image.open(mask_stems[stem]) as msk:            mask = np.array(msk)            if mask.ndim == 3:                mask = mask.max(axis=2)            binary = (mask > MASK_THRESHOLD).astype(np.uint8)            mask_pil = Image.fromarray(binary * 255, mode='L')            mask_resized = mask_pil.resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.NEAREST)            masks_arr[i] = (np.array(mask_resized) > 0).astype(np.uint8)        # Save cache    np.savez_compressed(        cache_path,        ids=np.array(common_stems, dtype=np.str_),        images=images_arr,        masks=masks_arr,    )        print('  Cache saved: ' + str(cache_path) + ' (' + '{:.1f}'.format(cache_path.stat().st_size / 1e6) + ' MB)')    return cache_path, common_stems# Build caches for all available datasetscache_dir = LOCAL_DATA_ROOT / 'cache'cache_dir.mkdir(parents=True, exist_ok=True)dataset_caches = {}for report in preparation_reports:    key = report['key']    if report['status'] == 'ok' and report['n_valid_pairs'] > 0:        info = DATASET_SOURCES[key]        dataset_dir = DRIVE_DATA_ROOT / key        try:            cache_path, ids = build_external_cache(key, info, dataset_dir, cache_dir)            dataset_caches[key] = {                'cache_path': cache_path,                'ids': ids,                'info': info,            }        except Exception as e:            print('Failed to build cache for ' + key + ': ' + str(e))    else:        print('Skipping ' + key + ': ' + report['status'])

## 7. Create Evaluation DataLoaders

In [ ]:
# Create DataLoaders for each external dataset using the cachefrom torch.utils.data import DataLoaderfrom src.datasets import PolypDatasetBATCH_SIZE = cfg['train']['batch_size']NUM_WORKERS = cfg['data']['num_workers']PIN_MEMORY = cfg['data']['pin_memory']dataloaders = {}dataset_ids = {}for key, cache_info in dataset_caches.items():    cache_path = cache_info['cache_path']    ids = cache_info['ids']    info = cache_info['info']        # Create a temporary split file with all IDs    split_file = LOCAL_DATA_ROOT / (key + '_all.txt')    with open(split_file, 'w') as f:        for id_ in ids:            f.write(id_ + '\n')        # Create dataset using the cache    dataset = PolypDataset(        split_file=split_file,        root=cache_info['cache_path'].parent.parent,  # dummy root        transform=None,        image_size=IMAGE_SIZE,        image_dir='images',  # not used with cache        mask_dir='masks',    # not used with cache        image_ext=info['image_ext'],        mask_ext=info['mask_ext'],        mask_suffix=info['mask_suffix'],        mask_threshold=MASK_THRESHOLD,        cache_path=str(cache_path),        normalize=NORMALIZE,        mean=cfg['data']['mean'],        std=cfg['data']['std'],    )        loader = DataLoader(        dataset,        batch_size=BATCH_SIZE,        shuffle=False,        num_workers=NUM_WORKERS,        pin_memory=PIN_MEMORY,        drop_last=False,    )        dataloaders[key] = loader    dataset_ids[key] = ids    print(key + ': ' + str(len(dataset)) + ' samples, ' + str(len(loader)) + ' batches')

## 8. Run Evaluation on External Datasets

In [ ]:
# Evaluation functionfrom src.metrics import SegMetricsfrom src.losses import build_lossimport timeimport csvloss_fn = build_loss(cfg).to(device)use_amp = cfg['train']['amp'] and device.type == 'cuda'def evaluate_dataset(model, loader, dataset_key, dataset_name):    """Evaluate model on a dataset and return per-image metrics."""    model.eval()    metrics = SegMetrics(from_logits=True, threshold=0.5)    total_loss = 0.0    batches = 0        print('Evaluating ' + dataset_name + '...')    start_time = time.time()        with torch.inference_mode():        for batch in tqdm(loader, desc=dataset_name):            images = batch['image'].to(device, non_blocking=True)            masks = batch['mask'].to(device, non_blocking=True)            ids = batch['id']                        if use_amp:                with torch.autocast(device_type='cuda', dtype=torch.float16):                    logits = model(images)                    loss = loss_fn(logits, masks)            else:                logits = model(images)                loss = loss_fn(logits, masks)                        total_loss += float(loss)            batches += 1            metrics.update(logits, masks, ids=list(ids))        elapsed = time.time() - start_time        # Compute summary    summary = metrics.compute()    per_image = metrics.per_image()        print('  ' + dataset_name + ': n=' + '{:.0f}'.format(summary['n']) + ', Dice={:.4f}'.format(summary['dice']) + ', IoU={:.4f}'.format(summary['iou']) + ', time={:.1f}s'.format(elapsed))        return {        'dataset_key': dataset_key,        'dataset_name': dataset_name,        'summary': summary,        'per_image': per_image,        'elapsed': elapsed,        'avg_loss': total_loss / max(1, batches),    }

In [ ]:
# Run evaluation on all datasetsevaluation_results = {}for key, loader in dataloaders.items():    info = DATASET_SOURCES[key]    result = evaluate_dataset(model, loader, key, info['name'])    evaluation_results[key] = result

In [ ]:
# Compute MAE for each datasetdef compute_mae(model, loader, device, use_amp):    """Compute Mean Absolute Error."""    model.eval()    total_mae = 0.0    total_pixels = 0        with torch.inference_mode():        for batch in loader:            images = batch['image'].to(device, non_blocking=True)            masks = batch['mask'].to(device, non_blocking=True)                        if use_amp:                with torch.autocast(device_type='cuda', dtype=torch.float16):                    logits = model(images)            else:                logits = model(images)                        probs = torch.sigmoid(logits)            mae = torch.abs(probs - masks).sum().item()            total_mae += mae            total_pixels += masks.numel()        return total_mae / max(1, total_pixels)for key in evaluation_results:    loader = dataloaders[key]    mae = compute_mae(model, loader, device, use_amp)    evaluation_results[key]['mae'] = mae    print(key + ' MAE: ' + '{:.6f}'.format(mae))

## 9. Generate Qualitative Visualizations

In [ ]:
# Visualization functionimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport numpy as npdef denormalize_image(image_tensor, mean, std):    """Denormalize image tensor for visualization."""    image = image_tensor.cpu().numpy()    mean = np.array(mean).reshape(3, 1, 1)    std = np.array(std).reshape(3, 1, 1)    image = image * std + mean    image = np.clip(image, 0, 1)    return image.transpose(1, 2, 0)def create_visualization(image, gt_mask, pred_mask, pred_prob, save_path, title):    """Create a 5-panel visualization."""    fig, axes = plt.subplots(1, 5, figsize=(20, 4))        # Original image    axes[0].imshow(image)    axes[0].set_title('Original Image')    axes[0].axis('off')        # Ground truth    axes[1].imshow(gt_mask, cmap='gray', vmin=0, vmax=1)    axes[1].set_title('Ground Truth')    axes[1].axis('off')        # Prediction (binary)    axes[2].imshow(pred_mask, cmap='gray', vmin=0, vmax=1)    axes[2].set_title('Prediction (thresh=0.5)')    axes[2].axis('off')        # GT Overlay    overlay_gt = image.copy()    overlay_gt[gt_mask > 0.5] = [1, 0, 0]  # Red for GT    axes[3].imshow(overlay_gt)    axes[3].set_title('GT Overlay (Red)')    axes[3].axis('off')        # Prediction Overlay    overlay_pred = image.copy()    overlay_pred[pred_mask > 0.5] = [0, 1, 0]  # Green for prediction    axes[4].imshow(overlay_pred)    axes[4].set_title('Pred Overlay (Green)')    axes[4].axis('off')        fig.suptitle(title, fontsize=14)    plt.tight_layout()    plt.savefig(save_path, dpi=150, bbox_inches='tight')    plt.close(fig)def select_representative_samples(per_image_results, n_samples=10):    """Select representative samples based on Dice scores."""    if not per_image_results:        return []        # Sort by Dice    sorted_results = sorted(per_image_results, key=lambda x: x['dice'])    n = len(sorted_results)        indices = []    # Best    indices.extend(range(max(0, n - n_samples//2), n))    # Worst    indices.extend(range(0, min(n_samples//2, n)))    # Middle    mid = n // 2    indices.extend(range(max(0, mid - n_samples//4), min(n, mid + n_samples//4)))        # Unique and sorted    indices = sorted(set(indices))[:n_samples]    return [sorted_results[i] for i in indices]

In [ ]:
# Generate visualizations for each datasetmean = cfg['data']['mean']std = cfg['data']['std']for key, result in evaluation_results.items():    info = DATASET_SOURCES[key]    loader = dataloaders[key]    per_image = result['per_image']        # Select representative samples    selected = select_representative_samples(per_image, n_samples=10)        if not selected:        continue        vis_dir = LOCAL_VISUALIZATIONS / key    vis_dir.mkdir(parents=True, exist_ok=True)        print('Generating visualizations for ' + info['name'] + '...')        # We need to re-run inference on selected samples to get images    # Create a mapping from id to batch data    id_to_batch = {}    for batch in loader:        for i, id_ in enumerate(batch['id']):            id_to_batch[id_] = {                'image': batch['image'][i],                'mask': batch['mask'][i],            }        for item in selected:        id_ = item['id']        if id_ not in id_to_batch:            continue                batch_data = id_to_batch[id_]        image_tensor = batch_data['image'].unsqueeze(0).to(device)        gt_mask = batch_data['mask'].squeeze(0).cpu().numpy()                with torch.inference_mode():            if use_amp:                with torch.autocast(device_type='cuda', dtype=torch.float16):                    logits = model(image_tensor)            else:                logits = model(image_tensor)                probs = torch.sigmoid(logits).squeeze().cpu().numpy()        pred_mask = (probs > 0.5).astype(np.float32)                image = denormalize_image(batch_data['image'], mean, std)                title = info['name'] + ' | ' + id_ + ' | Dice: ' + '{:.4f}'.format(item['dice']) + ' | IoU: ' + '{:.4f}'.format(item['iou'])        save_path = vis_dir / (id_ + '.png')        create_visualization(image, gt_mask, pred_mask, probs, save_path, title)        print('  Saved ' + str(len(selected)) + ' visualizations to ' + str(vis_dir))

## 10. Generate Quantitative Reports

In [ ]:
# Generate summary tableimport pandas as pdfrom datetime import datetimesummary_rows = []per_image_dfs = {}for key, result in evaluation_results.items():    info = DATASET_SOURCES[key]    summary = result['summary']    per_image = result['per_image']    mae = result['mae']        # Compute std    dice_values = [r['dice'] for r in per_image]    iou_values = [r['iou'] for r in per_image]        dice_mean = np.mean(dice_values) if dice_values else 0    dice_std = np.std(dice_values, ddof=1) if len(dice_values) > 1 else 0    iou_mean = np.mean(iou_values) if iou_values else 0    iou_std = np.std(iou_values, ddof=1) if len(iou_values) > 1 else 0        summary_rows.append({        'Dataset': info['name'],        'Images Evaluated': int(summary['n']),        'Dice Mean': dice_mean,        'Dice Std': dice_std,        'IoU Mean': iou_mean,        'IoU Std': iou_std,        'MAE': mae,    })        # Per-image DataFrame    df = pd.DataFrame(per_image)    df['dataset'] = key    per_image_dfs[key] = dfsummary_df = pd.DataFrame(summary_rows)print('\n=== External Evaluation Summary ===')print(summary_df.to_string(index=False))

In [ ]:
# Save summary CSVsummary_csv_path = LOCAL_METRICS / 'external_evaluation_summary.csv'summary_df.to_csv(summary_csv_path, index=False, float_format='%.6f')print('Summary CSV saved: ' + str(summary_csv_path))# Save per-image CSVsfor key, df in per_image_dfs.items():    csv_path = LOCAL_METRICS / (key + '_per_image.csv')    df.to_csv(csv_path, index=False, float_format='%.6f')    print('Per-image CSV saved: ' + str(csv_path))

In [ ]:
# Generate markdown summarymd_lines = [    '# External Dataset Evaluation Summary',    '',    '**Experiment:** ' + EXPERIMENT_NAME + '  ',    '**Checkpoint:** best.pt (epoch ' + str(checkpoint.get('epoch', '?')) + ')  ',    '**Git Commit:** ' + str(run_info.get('git_commit', 'unknown')) + '  ',    '**Evaluation Date:** ' + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + '  ',    '**Model Parameters:** ' + '{:.2f}M'.format(total_params/1e6) + '  ',    '**Input Resolution:** ' + str(IMAGE_SIZE) + 'x' + str(IMAGE_SIZE) + '  ',    '**Normalization:** ImageNet (mean=' + str(cfg['data']['mean']) + ', std=' + str(cfg['data']['std']) + ')  ',    '**Mask Threshold:** >' + str(MASK_THRESHOLD) + ' (before resize)  ',    '**Prediction Threshold:** 0.5 (after sigmoid)  ',    '**TTA:** OFF  ',    '','| Dataset | Images Evaluated | Dice Mean | Dice Std | IoU Mean | IoU Std | MAE |''|---|---:|---:|---:|---:|---:|---:|']for _, row in summary_df.iterrows():    md_lines.append(        '| ' + str(row['Dataset']) + ' | ' + str(int(row['Images Evaluated'])) + ' | '        + '{:.4f} | {:.4f} | '.format(row['Dice Mean'], row['Dice Std'])        + '{:.4f} | {:.4f} | {:.6f} |'.format(row['IoU Mean'], row['IoU Std'], row['MAE'])    )md_lines.extend([    '','## Per-Image Metrics',    '',    'Per-image CSV files are saved in the metrics directory.',    '','## Visualizations',    '',    'Qualitative visualizations (original, GT, prediction, GT overlay, pred overlay) are saved in the visualizations directory.',    '','## Notes',    '',    '- This is a zero-shot cross-dataset generalization evaluation.',    '- The model was trained on merged Kvasir-SEG + CVC-ClinicDB.',    '- No fine-tuning or retraining was performed on external datasets.',    '- Metrics are computed per-image then averaged (not pooled).',    '- Dice and IoU use threshold 0.5 on sigmoid probabilities.',    '- MAE is computed on raw probabilities.',    ''])md_content = '\n'.join(md_lines)md_path = LOCAL_METRICS / 'external_evaluation_summary.md'with open(md_path, 'w') as f:    f.write(md_content)print(md_content)

In [ ]:
# Save evaluation configeval_config = {    'experiment': EXPERIMENT_NAME,    'checkpoint': 'best.pt',    'checkpoint_epoch': checkpoint.get('epoch'),    'git_commit': run_info.get('git_commit'),    'git_dirty': run_info.get('git_dirty'),    'model_config': {        'encoder_a': str(cfg['model']['encoder_a']),        'encoder_b': str(cfg['model']['encoder_b']),        'fusion': str(cfg['model']['fusion']),        'channels': str(cfg['model']['channels']),        'fusion_heads': str(cfg['model']['fusion_heads']),        'pos_enc_2d': str(cfg['model']['pos_enc_2d']),        'msu': str(cfg['decoder']['msu']),        'msu_mode': str(cfg['decoder']['msu_mode']),    },    'preprocessing': {        'image_size': IMAGE_SIZE,        'normalize': NORMALIZE,        'mean': str(cfg['data']['mean']),        'std': str(cfg['data']['std']),        'mask_threshold': MASK_THRESHOLD,    },    'evaluation': {        'prediction_threshold': 0.5,        'tta': False,        'batch_size': BATCH_SIZE,        'device': str(device),    },    'datasets_evaluated': list(evaluation_results.keys()),    'timestamp': datetime.now().isoformat(),}eval_config_path = LOCAL_METRICS / 'evaluation_config.json'with open(eval_config_path, 'w') as f:    json.dump(eval_config, f, indent=2)print('Evaluation config saved: ' + str(eval_config_path))

In [ ]:
# Save environment infoenv_info = {    'torch_version': torch.__version__,    'cuda_available': torch.cuda.is_available(),    'cuda_version': torch.version.cuda if torch.cuda.is_available() else None,    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,    'python_version': sys.version,    'numpy_version': np.__version__,    'pillow_version': Image.__version__,    'matplotlib_version': matplotlib.__version__,    'timm_version': __import__('timm').__version__,    'timestamp': datetime.now().isoformat(),}env_path = LOCAL_METRICS / 'environment_info.json'with open(env_path, 'w') as f:    json.dump(env_info, f, indent=2)print('Environment info saved: ' + str(env_path))

In [ ]:
# Save evaluation loglog_lines = [    'External Dataset Evaluation Log - ' + datetime.now().isoformat(),    'Experiment: ' + EXPERIMENT_NAME,    'Checkpoint: best.pt (epoch ' + str(checkpoint.get('epoch', '?')) + ')',    'Git commit: ' + str(run_info.get('git_commit', 'unknown')),    'Device: ' + str(device),    'Model parameters: ' + '{:,}'.format(total_params),    '']for key, result in evaluation_results.items():    info = DATASET_SOURCES[key]    summary = result['summary']    mae = result['mae']    dice_values = [r['dice'] for r in result['per_image']]    iou_values = [r['iou'] for r in result['per_image']]        log_lines.extend([        '=== ' + info['name'] + ' (' + key + ') ===',        '  Images evaluated: ' + str(int(summary['n'])),        '  Dice: ' + '{:.4f} +/- {:.4f}'.format(np.mean(dice_values), np.std(dice_values, ddof=1)),        '  IoU: ' + '{:.4f} +/- {:.4f}'.format(np.mean(iou_values), np.std(iou_values, ddof=1)),        '  MAE: ' + '{:.6f}'.format(mae),        '  Evaluation time: ' + '{:.1f}s'.format(result['elapsed']),        '  Avg loss: ' + '{:.6f}'.format(result['avg_loss']),        ''    ])log_content = '\n'.join(log_lines)log_path = LOCAL_LOGS / 'evaluation.log'with open(log_path, 'w') as f:    f.write(log_content)print(log_content)

## 11. Generate Comparison Visualization

In [ ]:
# Create comparison bar chartfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))datasets = [DATASET_SOURCES[k]['name'] for k in evaluation_results.keys()]dice_means = [np.mean([r['dice'] for r in evaluation_results[k]['per_image']]) for k in evaluation_results.keys()]dice_stds = [np.std([r['dice'] for r in evaluation_results[k]['per_image']], ddof=1) if len(evaluation_results[k]['per_image']) > 1 else 0 for k in evaluation_results.keys()]iou_means = [np.mean([r['iou'] for r in evaluation_results[k]['per_image']]) for k in evaluation_results.keys()]iou_stds = [np.std([r['iou'] for r in evaluation_results[k]['per_image']], ddof=1) if len(evaluation_results[k]['per_image']) > 1 else 0 for k in evaluation_results.keys()]x = np.arange(len(datasets))width = 0.35# Dicebars1 = ax1.bar(x, dice_means, width, yerr=dice_stds, capsize=5, label='Dice', color='skyblue', edgecolor='black')ax1.set_ylabel('Dice Coefficient')ax1.set_title('Dice Coefficient by Dataset')ax1.set_xticks(x)ax1.set_xticklabels(datasets, rotation=15, ha='right')ax1.set_ylim(0, 1.0)ax1.grid(axis='y', alpha=0.3)for bar, val in zip(bars1, dice_means):    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, '{:.3f}'.format(val), ha='center', va='bottom', fontsize=10)# IoUbars2 = ax2.bar(x, iou_means, width, yerr=iou_stds, capsize=5, label='IoU', color='lightcoral', edgecolor='black')ax2.set_ylabel('IoU')ax2.set_title('IoU by Dataset')ax2.set_xticks(x)ax2.set_xticklabels(datasets, rotation=15, ha='right')ax2.set_ylim(0, 1.0)ax2.grid(axis='y', alpha=0.3)for bar, val in zip(bars2, iou_means):    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, '{:.3f}'.format(val), ha='center', va='bottom', fontsize=10)plt.suptitle(EXPERIMENT_NAME + ' - External Dataset Generalization', fontsize=14)plt.tight_layout()chart_path = LOCAL_VISUALIZATIONS / 'comparison_dice_iou.png'plt.savefig(chart_path, dpi=150, bbox_inches='tight')plt.close(fig)print('Comparison chart saved: ' + str(chart_path))

## 12. Copy All Results to Google Drive

In [ ]:
# Create timestamped run directory on Driverun_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')drive_run_dir = DRIVE_OUTPUT_DIR / ('eval_' + run_timestamp)drive_run_dir.mkdir(parents=True, exist_ok=True)import shutil# Copy metricsdrive_metrics = drive_run_dir / 'metrics'shutil.copytree(LOCAL_METRICS, drive_metrics)print('Copied metrics to ' + str(drive_metrics))# Copy visualizationsdrive_vis = drive_run_dir / 'visualizations'shutil.copytree(LOCAL_VISUALIZATIONS, drive_vis)print('Copied visualizations to ' + str(drive_vis))# Copy logsdrive_logs = drive_run_dir / 'logs'shutil.copytree(LOCAL_LOGS, drive_logs)print('Copied logs to ' + str(drive_logs))# Copy preparation reportshutil.copy2(LOCAL_METRICS / 'dataset_preparation_report.json', drive_run_dir / 'dataset_preparation_report.json')shutil.copy2(LOCAL_METRICS / 'evaluation_config.json', drive_run_dir / 'evaluation_config.json')shutil.copy2(LOCAL_METRICS / 'environment_info.json', drive_run_dir / 'environment_info.json')shutil.copy2(LOCAL_METRICS / 'external_evaluation_summary.md', drive_run_dir / 'external_evaluation_summary.md')shutil.copy2(LOCAL_METRICS / 'external_evaluation_summary.csv', drive_run_dir / 'external_evaluation_summary.csv')shutil.copy2(LOCAL_LOGS / 'evaluation.log', drive_run_dir / 'evaluation.log')print('
All results saved to: ' + str(drive_run_dir))

## 13. Final Integrity Checks

In [ ]:
# Final integrity checksprint('=== Final Integrity Checks ===')# 1. Checkpoint exists and loadsassert (EXPERIMENT_DIR / 'best.pt').exists(), 'Checkpoint missing'print('Checkpoint exists')# 2. Model architecture matches configassert total_params == run_info.get('params_total'), 'Parameter count mismatch'print('Model architecture matches config')# 3. All weights loadedassert len(missing_keys) == 0, 'Missing keys in checkpoint'assert len(unexpected_keys) == 0, 'Unexpected keys in checkpoint'print('All weights loaded correctly')# 4. GPU availableif torch.cuda.is_available():    print('GPU available: ' + torch.cuda.get_device_name(0))else:    print('Running on CPU (evaluation will be slower)')# 5. Every dataset has valid pairsfor key, result in evaluation_results.items():    n = int(result['summary']['n'])    assert n > 0, 'Dataset ' + key + ' has no valid samples'    print(key + ': ' + str(n) + ' samples evaluated')# 6. Images and masks have compatible dimensionsfor key, loader in dataloaders.items():    batch = next(iter(loader))    assert batch['image'].shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE), key + ': image shape mismatch'    assert batch['mask'].shape[1:] == (1, IMAGE_SIZE, IMAGE_SIZE), key + ': mask shape mismatch'print('Image/mask dimensions compatible')# 7. Model output has expected shapewith torch.inference_mode():    test_out = model(torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device))assert test_out.shape == (1, 1, IMAGE_SIZE, IMAGE_SIZE), 'Output shape mismatch: ' + str(test_out.shape)print('Model output shape correct')# 8. Predictions contain valid probabilitiestest_probs = torch.sigmoid(test_out)assert test_probs.min() >= 0 and test_probs.max() <= 1, 'Probabilities out of range'print('Predictions in valid range [0, 1]')# 9. Metrics are finitefor key, result in evaluation_results.items():    dice_vals = [r['dice'] for r in result['per_image']]    iou_vals = [r['iou'] for r in result['per_image']]    assert all(np.isfinite(dice_vals)), key + ': non-finite Dice values'    assert all(np.isfinite(iou_vals)), key + ': non-finite IoU values'print('All metrics finite')# 10. Every image evaluated exactly oncefor key, result in evaluation_results.items():    ids = [r['id'] for r in result['per_image']]    assert len(ids) == len(set(ids)), key + ': duplicate image IDs found'    assert len(ids) == len(dataset_ids[key]), key + ': not all images evaluated'print('Every image evaluated exactly once')print('
All integrity checks passed!')

## 14. Evaluation Summary

In [ ]:
# Print final summaryprint('=' * 60)print('EXTERNAL DATASET EVALUATION COMPLETE: ' + EXPERIMENT_NAME)print('=' * 60)print('Checkpoint: best.pt (epoch ' + str(checkpoint.get('epoch', '?')) + ')')print('Git commit: ' + str(run_info.get('git_commit', 'unknown')))print('Device: ' + str(device))print('Model parameters: ' + '{:.2f}M'.format(total_params/1e6))print('')print('Results:')for key, result in evaluation_results.items():    info = DATASET_SOURCES[key]    dice_vals = [r['dice'] for r in result['per_image']]    iou_vals = [r['iou'] for r in result['per_image']]    mae = result['mae']    print('  ' + '{:25s}'.format(info['name']) + ' n=' + '{:4d}'.format(int(result['summary']['n'])) + '  '          + 'Dice=' + '{:.4f}+/-{:.4f}  '.format(np.mean(dice_vals), np.std(dice_vals, ddof=1))          + 'IoU=' + '{:.4f}+/-{:.4f}  '.format(np.mean(iou_vals), np.std(iou_vals, ddof=1))          + 'MAE=' + '{:.6f}'.format(mae))print('')print('Results saved to: ' + str(drive_run_dir))print('')print('Files generated:')print('  - external_evaluation_summary.csv (quantitative summary)')print('  - external_evaluation_summary.md (markdown report)')print('  - colondb_per_image.csv, etis_per_image.csv, cvc300_per_image.csv')print('  - dataset_preparation_report.json')print('  - evaluation_config.json')print('  - environment_info.json')print('  - evaluation.log')print('  - comparison_dice_iou.png')print('  - visualizations/<dataset>/*.png (10 per dataset)')print('=' * 60)